In [64]:
import pandas as pd

results = pd.read_csv("./outputs/tables/results_all.csv")
summary = pd.read_csv("./outputs/tables/table1_summary.csv")
fairness = pd.read_csv("./outputs/tables/table2_fairness_pivot.csv")

# Table 1

In [67]:
import pandas as pd
import plotly.graph_objects as go

# ── mappings ─────────────────────────────────────────────
dataset_map = {
    "D0": "Balanced",
    "D1A": "Row Removal",
    "D2A": "Missingness"
}

model_map = {
    "liu_glm": "GLM",
    "liu_xgboost": "XGBoost",
    "liu_rnn": "RNN"
}

# ── clean + filter ───────────────────────────────────────
df = results.copy()

df["dataset_id"] = df["dataset_id"].map(dataset_map)
df["model"] = df["model"].map(model_map)

df = df.dropna(subset=["dataset_id", "model"])

df = df[df["mitigation"] == "none"]

# ── select + rename ──────────────────────────────────────
df = df[[
    "model", "dataset_id",
    "overall_physionet_utility",
    "disparate_impact",
    "equal_opportunity"
]].rename(columns={
    "model": "Model",
    "dataset_id": "Dataset",
    "overall_physionet_utility": "Utility",
    "disparate_impact": "Disparate Impact",
    "equal_opportunity": "Equal Opportunity",
})

df = df.round(3)

# ── enforce ordering ─────────────────────────────────────
model_order = ["GLM", "RNN", "XGBoost"]
dataset_order = ["Balanced", "Row Removal", "Missingness"]

df["Model"] = pd.Categorical(df["Model"], model_order)
df["Dataset"] = pd.Categorical(df["Dataset"], dataset_order)

df = df.sort_values(["Model", "Dataset"])

# ── build grouped rows (blank model cells) ───────────────
rows = []

for model in model_order:
    df_m = df[df["Model"] == model]

    for i, (_, r) in enumerate(df_m.iterrows()):
        rows.append({
            "Model": model if i == 0 else "",
            "Dataset": r["Dataset"],
            "Utility": r["Utility"],
            "Disparate Impact": r["Disparate Impact"],
            "Equal Opportunity": r["Equal Opportunity"],
        })

table_df = pd.DataFrame(rows)

# ── plotly table ─────────────────────────────────────────
fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(table_df.columns),
        fill_color="lightgrey",
        align="left",
        font=dict(size=16)
    ),
    cells=dict(
        values=[table_df[c] for c in table_df.columns],
        align="left",
        height=40
    )
)])

fig.update_layout(
    title="Baseline Model Performance (No Mitigation)",
    height=650
)

fig.show()

# Table 2

In [ ]:
import pandas as pd
import plotly.graph_objects as go

def make_model_table(results, model_name):

    dataset_map = {
        "D0": "Balanced",
        "D1A": "Row Removal",
        "D2A": "Missingness"
    }

    model_map = {
        "liu_glm": "GLM",
        "liu_xgboost": "XGBoost",
        "liu_rnn": "RNN"
    }

    mitigation_map = {
        "none": "Original",
        "reweighting": "Reweighting",
        "smote": "SMOTE",
        "threshold_optimization": "Fairness Penalty"
    }

    df = results.copy()

    df["Dataset"] = df["dataset_id"].map(dataset_map)
    df["Model"] = df["model"].map(model_map)
    df["Mitigation"] = df["mitigation"].map(mitigation_map)

    df = df.dropna(subset=["Dataset", "Model", "Mitigation"])
    df = df[df["Model"] == model_name]

    df = df[[
        "Mitigation", "Dataset",
        "overall_physionet_utility",
        "disparate_impact",
        "equal_opportunity"
    ]].rename(columns={
        "overall_physionet_utility": "Utility",
        "disparate_impact": "Disparate Impact",
        "equal_opportunity": "Equal Opportunity",
    }).round(3)

    mitigation_order = ["Original", "Reweighting", "SMOTE", "Fairness Penalty"]
    dataset_order = ["Balanced", "Row Removal", "Missingness"]

    df["Mitigation"] = pd.Categorical(df["Mitigation"], mitigation_order)
    df["Dataset"] = pd.Categorical(df["Dataset"], dataset_order)

    df = df.sort_values(["Dataset", "Mitigation"])

    # ── NEW grouping logic ───────────────────────
    rows = []

    for dataset in dataset_order:
        df_ds = df[df["Dataset"] == dataset]

        for mitigation in mitigation_order:
            df_mit = df_ds[df_ds["Mitigation"] == mitigation]

            for i, (_, r) in enumerate(df_mit.iterrows()):
                rows.append({
                    "Dataset": dataset if (mitigation == mitigation_order[0] and i == 0) else "",
                    "Mitigation": mitigation if i == 0 else "",
                    "Utility": r["Utility"],
                    "Disparate Impact": r["Disparate Impact"],
                    "Equal Opportunity": r["Equal Opportunity"],
                })

    table_df = pd.DataFrame(rows)

    fig = go.Figure(data=[go.Table(
        header=dict(
            values=list(table_df.columns),
            fill_color="lightgrey",
            align="left",
            font=dict(size=16)
        ),
        cells=dict(
            values=[table_df[c] for c in table_df.columns],
            align="left",
            height=35
        )
    )])

    fig.update_layout(
        title=f"{model_name} — Dataset-first Comparison",
        height=800
    )

    print("\n", model_name)
    print(table_df.to_string(index=False))

    return fig
figs = {}

for m in ["GLM", "RNN", "XGBoost"]:
    figs[m] = make_model_table(results, m)

for f in figs.values():
    f.show()



 GLM
    Dataset       Mitigation  Utility  Disparate Impact  Equal Opportunity
   Balanced         Original    0.147             0.877              0.096
                 Reweighting    0.130             0.871              0.119
                       SMOTE    0.145             0.884              0.096
            Fairness Penalty   -0.054             0.999              0.007
Row Removal         Original    0.305             0.913              0.025
                 Reweighting    0.304             0.899              0.028
                       SMOTE    0.327             0.899              0.006
            Fairness Penalty    0.137             1.139              0.023
Missingness         Original    0.148             0.878              0.104
                 Reweighting    0.127             0.873              0.127
                       SMOTE    0.145             0.881              0.104
            Fairness Penalty   -0.054             0.999              0.007

 RNN
    Dataset  

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

def make_model_view(results, model_name, mode="table", metric="Utility", output_dir=None):

    dataset_map = {
        "D0": "Balanced",
        "D1A": "Row Removal",
        "D2A": "Missingness"
    }

    model_map = {
        "liu_glm": "GLM",
        "liu_xgboost": "XGBoost",
        "liu_rnn": "RNN"
    }

    mitigation_map = {
        "none": "Original",
        "reweighting": "Reweighting",
        "smote": "SMOTE",
        "threshold_optimization": "Fairness Penalty"
    }

    df = results.copy()

    df["Dataset"] = df["dataset_id"].map(dataset_map)
    df["Model"] = df["model"].map(model_map)
    df["Mitigation"] = df["mitigation"].map(mitigation_map)

    df = df.dropna(subset=["Dataset", "Model", "Mitigation"])
    df = df[df["Model"] == model_name]

    df = df[[
        "Mitigation", "Dataset",
        "overall_physionet_utility",
        "disparate_impact",
        "equal_opportunity"
    ]].rename(columns={
        "overall_physionet_utility": "Utility",
        "disparate_impact": "Disparate Impact",
        "equal_opportunity": "Equal Opportunity",
    }).round(3)

    mitigation_order = ["Original", "Reweighting", "SMOTE", "Fairness Penalty"]
    dataset_order = ["Balanced", "Row Removal", "Missingness"]

    df["Mitigation"] = pd.Categorical(df["Mitigation"], mitigation_order)
    df["Dataset"] = pd.Categorical(df["Dataset"], dataset_order)

    df = df.sort_values(["Dataset", "Mitigation"])

    # =========================
    # TABLE MODE
    # =========================
    if mode == "table":

        rows = []

        for dataset in dataset_order:
            df_ds = df[df["Dataset"] == dataset]

            for mitigation in mitigation_order:
                df_mit = df_ds[df_ds["Mitigation"] == mitigation]

                for i, (_, r) in enumerate(df_mit.iterrows()):
                    rows.append({
                        "Dataset": dataset if (mitigation == mitigation_order[0] and i == 0) else "",
                        "Mitigation": mitigation if i == 0 else "",
                        "Utility": r["Utility"],
                        "Disparate Impact": r["Disparate Impact"],
                        "Equal Opportunity": r["Equal Opportunity"],
                    })

        table_df = pd.DataFrame(rows)

        fig = go.Figure(data=[go.Table(
            header=dict(
                values=list(table_df.columns),
                fill_color="lightgrey",
                align="left",
                font=dict(size=16)
            ),
            cells=dict(
                values=[table_df[c] for c in table_df.columns],
                align="left",
                height=35
            )
        )])

        fig.update_layout(
            title=f"{model_name} — Dataset-first Table",
            height=800
        )

    # =========================
    # HEATMAP MODE
    # =========================
    elif mode == "heatmap":

        pivot = df.pivot_table(
            index="Dataset",
            columns="Mitigation",
            values=metric,
            aggfunc="mean"
        )

        fig = px.imshow(
            pivot,
            text_auto=True,
            aspect="auto"
        )

        fig.update_layout(
            title=f"{model_name} — {metric",
            xaxis_title="Mitigation",
            yaxis_title="Dataset"
        )

    else:
        raise ValueError("mode must be 'table' or 'heatmap'")

    # =========================
    # SAVE OUTPUT
    # =========================
    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        fname = f"{model_name}_{mode}"
        if mode == "heatmap":
            fname += f"_{metric}"

        # try static image first
        try:
            fig.write_image(output_dir / f"{fname}.png", scale=2)
        except Exception:
            # fallback to HTML if kaleido not installed
            fig.write_html(output_dir / f"{fname}.html")

    return fig

In [91]:
output_dir = "./outputs/figures"

for m in ["GLM", "RNN", "XGBoost"]:
    # make_model_view(results, m, mode="table", output_dir=output_dir).show()
    make_model_view(results, m, mode="heatmap", metric="Utility", output_dir=output_dir).show()
    make_model_view(results, m, mode="heatmap", metric="Equal Opportunity", output_dir=output_dir).show()